In [ ]:
import torch, shutil

assert torch.cuda.is_available(), "No GPU — Runtime > Change runtime type > GPU"

gpu = torch.cuda.get_device_properties(0)
print(f"GPU : {gpu.name}  ({gpu.total_memory/1e9:.1f} GB VRAM)")
print(f"CUDA: {torch.version.cuda} | PyTorch: {torch.__version__}")

In [ ]:
from google.colab import drive

drive.mount("/content/gdrive")

GDRIVE_ROOT = "/content/gdrive/MyDrive/EK100_MIR"
print(f"Drive mounted: {GDRIVE_ROOT}")

In [ ]:
%%bash

pip install -q pandas numpy tqdm scipy scikit-learn spacy
python -m spacy download en_core_web_sm -q 2>/dev/null || true

echo "Dependencies OK"

In [ ]:
%%bash

cd /content

if [ ! -d "MI-MM" ]; then
    git clone -q --depth 1 https://github.com/adrianofragomeni/MI-MM.git
    echo "Cloned: MI-MM"
else
    echo "Exists: MI-MM"
fi

if [ ! -d "Joint-Part-of-Speech-Embeddings" ]; then
    git clone -q --depth 1 https://github.com/mwray/Joint-Part-of-Speech-Embeddings.git
    echo "Cloned: Joint-Part-of-Speech-Embeddings"
else
    echo "Exists: Joint-Part-of-Speech-Embeddings"
fi

In [ ]:
import subprocess, re, pickle, shutil, zipfile, tempfile
from pathlib import Path
import numpy as np

GDRIVE_DATA = Path(f"{GDRIVE_ROOT}/data")
MIMM_DIR  = Path("/content/MI-MM")
JPOSE_DIR = Path("/content/Joint-Part-of-Speech-Embeddings")

In [ ]:
MIMM_DATA_DIR   = MIMM_DIR / "data"
MIMM_OUTPUT_DIR = MIMM_DIR / "output"
MIMM_OUTPUT_DIR.mkdir(exist_ok=True)

In [ ]:
MODELS_DIR     = JPOSE_DIR / "data" / "models"
VID_FEAT_DIR   = JPOSE_DIR / "data" / "video_features"
TXT_FEAT_DIR   = JPOSE_DIR / "data" / "text_features"
DATAFRAMES_DIR = JPOSE_DIR / "data" / "dataframes"
RELATIONAL_DIR = JPOSE_DIR / "data" / "relational"
RELEVANCY_DIR  = JPOSE_DIR / "data" / "relevancy"

In [ ]:
SUBMISSIONS_DIR = Path(f"{GDRIVE_ROOT}/submissions")
ZIPS_DIR        = Path(f"{GDRIVE_ROOT}/submission_zips")
SUBMISSIONS_DIR.mkdir(parents=True, exist_ok=True)
ZIPS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
print("Paths configured:")
print(f"  MI-MM      : {MIMM_DIR}")
print(f"  JPoSE      : {JPOSE_DIR}")
print(f"  Drive data : {GDRIVE_DATA}")
print(f"  Submissions: {SUBMISSIONS_DIR}")
print(f"  ZIPs       : {ZIPS_DIR}")

In [ ]:
JPOSE_ZIP = GDRIVE_DATA / "JPoSE_data.zip"

In [ ]:
def data_ready():

    checkpoint = MODELS_DIR / "JPoSE_BEST" / "model" / "EPIC_100_retrieval_JPoSE_BEST.pth"
    return (
        checkpoint.exists()
        and VID_FEAT_DIR.exists() and any(VID_FEAT_DIR.iterdir())
        and TXT_FEAT_DIR.exists() and any(TXT_FEAT_DIR.iterdir())
    )

In [ ]:
if data_ready():
    print("JPoSE data: OK (already extracted)")

else:
    assert JPOSE_ZIP.exists(), (
        f"Not found: {JPOSE_ZIP}\n"
        "Run colab_download_data.ipynb first."
    )

    print(f"Extracting {JPOSE_ZIP.name} ({JPOSE_ZIP.stat().st_size/1e9:.2f} GB)...")
    
    subprocess.run(
        f'unzip -o -q "{JPOSE_ZIP}" -d "{JPOSE_DIR}"',
        shell=True, check=True,
    )
    print("JPoSE extracted OK")

In [ ]:
MIMM_ZIP = GDRIVE_DATA / "MI-MM_data.zip"

In [ ]:
def data_ready():

    mimm_dataframes = MIMM_DATA_DIR / "dataframes"
    mimm_ready = mimm_dataframes.is_dir() and any(mimm_dataframes.rglob("*"))

    return mimm_ready

In [ ]:
if data_ready():
    print("MI-MM data : OK (already extracted)")

else:
    assert MIMM_ZIP.exists(), (
        f"Not found: {MIMM_ZIP}\n"
        "Run colab_download_data.ipynb first."
    )
    print(f"Extracting {MIMM_ZIP.name} ({MIMM_ZIP.stat().st_size/1e9:.2f} GB)...")

    with tempfile.TemporaryDirectory() as tmp:
        with zipfile.ZipFile(MIMM_ZIP, "r") as zf:
            entries    = zf.namelist()
            inner_zips = [e for e in entries if e.endswith(".zip")]
            if inner_zips:
                zf.extractall(tmp)
                src_zip = str(Path(tmp) / inner_zips[0])
                print(f"  Inner zip: {inner_zips[0]}")
            else:
                src_zip = str(MIMM_ZIP)

        if MIMM_DATA_DIR.is_dir():
            shutil.rmtree(MIMM_DATA_DIR)
        MIMM_DATA_DIR.mkdir(parents=True, exist_ok=True)
        subprocess.run(
            f'unzip -o -q "{src_zip}" -d "{MIMM_DATA_DIR}"',
            shell=True, check=True,
        )

    print("MI-MM extracted OK")

In [ ]:
def check_path(path, label, required=True):
    p = Path(path)
    if p.is_dir():
        files  = [f for f in p.rglob("*") if f.is_file()]
        ok     = bool(files)
        detail = f"{len(files)} file(s)  ({sum(f.stat().st_size for f in files)/1e6:.0f} MB)"
    else:
        ok     = p.exists()
        detail = f"{p.stat().st_size/1e6:.0f} MB" if ok else "NOT FOUND"
    icon = "OK" if ok else ("ERR" if required else "WARN")
    print(f"  [{icon}]  {label}: {detail}")
    return ok

In [ ]:
print("── MI-MM ────────────────────────────────────────────────")
mimm_ok = all([
    check_path(MIMM_DATA_DIR / "dataframes", "dataframes"),
    check_path(MIMM_DATA_DIR / "features",   "S3D features"),
    check_path(MIMM_DATA_DIR / "models",     "models"),
    check_path(MIMM_DATA_DIR / "relevancy",  "relevancy"),
    check_path(MIMM_DATA_DIR / "resources",  "resources"),
])

In [ ]:
print("\n── JPoSE / MLP ──────────────────────────────────────────")
jpose_ok = all([
    check_path(MODELS_DIR,     "models"),
    check_path(VID_FEAT_DIR,   "video_features"),
    check_path(TXT_FEAT_DIR,   "text_features"),
    check_path(DATAFRAMES_DIR, "dataframes"),
    check_path(RELATIONAL_DIR, "relational"),
    check_path(RELEVANCY_DIR,  "relevancy"),
])

In [ ]:
check_path(MODELS_DIR / "JPoSE_BEST" / "model" / "EPIC_100_retrieval_JPoSE_BEST.pth", "JPoSE checkpoint")
check_path(MODELS_DIR / "MMEN_BEST"  / "model" / "EPIC_100_retrieval_MLP_BEST.pth",   "MLP  checkpoint")

if not (mimm_ok and jpose_ok):
    raise RuntimeError("Missing data — run colab_download_data.ipynb first")
print("\nAll data OK")

In [ ]:
def patch_file(path, old, new):
    p = Path(path)
    if not p.exists():
        return
    content = p.read_text()
    if old in content:
        p.write_text(content.replace(old, new))
        print(f"  patched : {p.name}")

In [ ]:
patch_file(
    MIMM_DIR / "src" / "loader" / "loader_features.py",
    "import pickle5 as pickle",
    "import pickle",
)

In [ ]:
patch_file(
    MIMM_DIR / "src" / "testing.py",
    "best_model = th.load(Path(args_.path_model)/args_.best_model)",
    "best_model = th.load(Path(args_.path_model)/args_.best_model, weights_only=False)",
)

In [ ]:
patch_file(
    MIMM_DIR / "src" / "models" / "embedding_projection.py",
    'pretrained_dict = th.load(self.path_resources / "s3d_howto100m.pth")',
    'pretrained_dict = th.load(self.path_resources / "s3d_howto100m.pth", weights_only=False)',
)

In [ ]:
for fpath in (JPOSE_DIR / "src").rglob("*.py"):
    txt = fpath.read_text()
    new_txt = re.sub(
        r"torch\.load\(([^,)]+)\)",
        r"torch.load(\1, weights_only=False)",
        txt,
    )
    if new_txt != txt:
        fpath.write_text(new_txt)
        print(f"  patched : {fpath.name}")

print("Patches OK")

In [ ]:
print("Running MI-MM inference ...")
r = subprocess.run(
    f'cd "{MIMM_DIR}/src" && python testing.py 2>&1',
    shell=True, capture_output=True, text=True, timeout=900,
)

for line in r.stdout.splitlines():
    if any(k in line for k in ["load", "Best", "Evaluat", "Error", "Traceback", "nDCG", "mAP"]):
        print(line)

if r.returncode != 0:
    print("\nFull output:", r.stdout[-1000:])
    print("STDERR:", r.stderr[-400:])
    raise RuntimeError("MI-MM inference failed")

mimm_raw = MIMM_OUTPUT_DIR / "test.pkl"
assert mimm_raw.exists(), f"Output not generated: {mimm_raw}"

with open(mimm_raw, "rb") as f:
    sub_mimm = pickle.load(f)

sim_mimm = np.array(sub_mimm["sim_mat"], dtype=np.float32)
assert sim_mimm.shape == (9668, 3842), f"Unexpected shape: {sim_mimm.shape}"

dst = SUBMISSIONS_DIR / "MI-MM_test_latest.pkl"
shutil.copy2(mimm_raw, dst)
print(f"\nMI-MM OK  sim_mat={sim_mimm.shape}  saved -> {dst.name}")

In [ ]:
MODEL_NAME = "JPoSE_BEST"
COMB_FUNC  = "cat"

CHECKPOINT = MODELS_DIR / MODEL_NAME / "model" / f"EPIC_100_retrieval_{MODEL_NAME}.pth"
assert CHECKPOINT.exists(), f"Checkpoint not found: {CHECKPOINT}"

jpose_out = SUBMISSIONS_DIR / f"{MODEL_NAME}_test_latest.pkl"

print(f"Running JPoSE inference ({MODEL_NAME}, comb-func={COMB_FUNC}) ...")
r = subprocess.run(
    f'cd "{JPOSE_DIR}" && PYTHONPATH="{JPOSE_DIR}/src" '
    f'python -W ignore src/train/test_jpose_triplet.py "{CHECKPOINT}" '
    f'--comb-func {COMB_FUNC} --challenge-submission "{jpose_out}" --gpu True 2>&1',
    shell=True, capture_output=True, text=True, timeout=600,
)

print(r.stdout[-2000:])
if r.returncode != 0:
    print("STDERR:", r.stderr[-400:])
    raise RuntimeError("JPoSE inference failed")

assert jpose_out.exists(), f"Output not generated: {jpose_out}"
with open(jpose_out, "rb") as f:
    sub_jpose = pickle.load(f)

sim_jpose = np.array(sub_jpose["sim_mat"], dtype=np.float32)
assert sim_jpose.shape == (9668, 3842), f"Unexpected shape: {sim_jpose.shape}"
print(f"\nJPoSE OK  sim_mat={sim_jpose.shape}  saved -> {jpose_out.name}")

In [ ]:
MMEN_MODEL = MODELS_DIR / "MMEN_BEST" / "model" / "EPIC_100_retrieval_MLP_BEST.pth"
assert MMEN_MODEL.exists(), f"Checkpoint not found: {MMEN_MODEL}"

mmen_out = SUBMISSIONS_DIR / "MMEN_BEST_test_latest.pkl"

print("Running MLP/MMEN inference ...")
r = subprocess.run(
    f'cd "{JPOSE_DIR}" && PYTHONPATH="{JPOSE_DIR}/src" '
    f'python -W ignore src/train/test_mmen_triplet.py "{MMEN_MODEL}" '
    f'--challenge-submission "{mmen_out}" --gpu True 2>&1',
    shell=True, capture_output=True, text=True, timeout=600,
)

print(r.stdout[-2000:])
if r.returncode != 0:
    print("STDERR:", r.stderr[-400:])
    raise RuntimeError("MLP/MMEN inference failed")

assert mmen_out.exists(), f"Output not generated: {mmen_out}"
with open(mmen_out, "rb") as f:
    sub_mmen = pickle.load(f)

sim_mlp = np.array(sub_mmen["sim_mat"], dtype=np.float32)
assert sim_mlp.shape == (9668, 3842), f"Unexpected shape: {sim_mlp.shape}"
print(f"\nMLP OK  sim_mat={sim_mlp.shape}  saved -> {mmen_out.name}")

In [ ]:
def normalize_sim(sim):
    """Min-max normalise each row (query) to [0, 1]."""
    mn = sim.min(axis=1, keepdims=True)
    mx = sim.max(axis=1, keepdims=True)
    return (sim - mn) / (mx - mn + 1e-9)

assert list(sub_jpose["vis_ids"]) == list(sub_mimm["vis_ids"]),  "vis_ids mismatch MI-MM vs JPoSE"
assert list(sub_jpose["txt_ids"]) == list(sub_mimm["txt_ids"]),  "txt_ids mismatch MI-MM vs JPoSE"
assert list(sub_jpose["vis_ids"]) == list(sub_mmen["vis_ids"]),  "vis_ids mismatch MLP  vs JPoSE"
assert list(sub_jpose["txt_ids"]) == list(sub_mmen["txt_ids"]),  "txt_ids mismatch MLP  vs JPoSE"

In [ ]:
W_MIMM, W_JPOSE, W_MLP = 1/3, 1/3, 1/3   # tune here if you have a val set

sim_ensemble = (
    W_MIMM  * normalize_sim(sim_mimm)  +
    W_JPOSE * normalize_sim(sim_jpose) +
    W_MLP   * normalize_sim(sim_mlp)
)

print(f"Ensemble shape  : {sim_ensemble.shape}")
print(f"Weights         : MI-MM={W_MIMM:.2f}  JPoSE={W_JPOSE:.2f}  MLP={W_MLP:.2f}")
print(f"Score range     : [{sim_ensemble.min():.4f}, {sim_ensemble.max():.4f}]")

In [ ]:
from scipy.sparse import csr_matrix

def rerank(sim_mat, k=20, alpha=0.3):
    S = sim_mat.astype(np.float64)

    S_v = S   / (np.linalg.norm(S,   axis=1, keepdims=True) + 1e-9)
    S_t = S.T / (np.linalg.norm(S.T, axis=1, keepdims=True) + 1e-9)
    A_vv = S_v @ S_v.T
    A_tt = S_t @ S_t.T

    def to_sparse_topk(A, k):
        n    = A.shape[0]
        idx  = np.argpartition(A, -k, axis=1)[:, -k:]
        vals = np.take_along_axis(A, idx, axis=1)
        vals = vals / (vals.sum(axis=1, keepdims=True) + 1e-9)
        rows = np.repeat(np.arange(n), k)
        return csr_matrix((vals.ravel(), (rows, idx.ravel())), shape=(n, n))

    A_vv_sp = to_sparse_topk(A_vv, k)
    A_tt_sp = to_sparse_topk(A_tt, k)
    del A_vv, A_tt

    S_exp = 0.5 * (A_vv_sp @ S + (A_tt_sp @ S.T).T)
    return ((1 - alpha) * S + alpha * S_exp).astype(np.float32)

In [ ]:
print("Re-ranking ensemble ...")
sim_reranked = rerank(sim_ensemble, k=20, alpha=0.3)

print(f"Re-ranked shape : {sim_reranked.shape}")
print(f"Parameters      : k=20  alpha=0.3")
print(f"Score range     : [{sim_reranked.min():.4f}, {sim_reranked.max():.4f}]")

In [ ]:
SLS_PT, SLS_TL, SLS_TD = 2, 3, 3

def make_compat_pickle(sim, vis_ids, txt_ids):
    payload = {
        "version":   "0.1",
        "challenge": "multi_instance_retrieval",
        "sls_pt": SLS_PT, "sls_tl": SLS_TL, "sls_td": SLS_TD,
        "sim_mat": np.array(sim, dtype=np.float32),
        "vis_ids": [str(v) for v in vis_ids],
        "txt_ids": [str(t) for t in txt_ids],
    }
    raw = pickle.dumps(payload, protocol=2)
    return raw.replace(b"numpy._core.multiarray", b"numpy.core.multiarray")

vis_ids = sub_jpose["vis_ids"]
txt_ids = sub_jpose["txt_ids"]
tmp_pkl = Path("/tmp/test.pkl")

submissions = [
    ("reranked",  sim_reranked),
    ("ensemble",  sim_ensemble),
    ("JPoSE",     sim_jpose),
    ("MI-MM",     sim_mimm),
    ("MLP",       sim_mlp),
]

print("Creating ZIPs:\n")
for name, sim in submissions:
    assert sim.shape == (9668, 3842)
    tmp_pkl.write_bytes(make_compat_pickle(sim, vis_ids, txt_ids))
    zip_path = ZIPS_DIR / f"{name}_submission.zip"
    subprocess.run(
        f'cd /tmp && zip -j "{zip_path}" test.pkl',
        shell=True, check=True, capture_output=True,
    )
    print(f"  {name:10s}  {zip_path.name}  ({zip_path.stat().st_size/1e6:.1f} MB)")

print(f"\nAll ZIPs saved to: {ZIPS_DIR}")
print("\nUpload to: https://www.codabench.org/competitions/12008")